In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from planar_ik import (
    ee_errors,
    fit_linear,
    fit_ridge,
    forward_kinematics,
    inverse_kinematics,
    mean_error_calc,
    polynomial_features,
    predict_linear,
    predict_ridge,
    print_summary,
    standardize_train_val_test,
    summarize_errors,
)


In [ ]:
# forward_kinematics is imported from planar_ik.geometry.


In [ ]:
# inverse_kinematics is imported from planar_ik.geometry.


In [ ]:
def error(x_true, y_true, L1=100, L2=100, elbow="up"):
    theta1, theta2 = inverse_kinematics(x_true, y_true, L1, L2, elbow=elbow)
    x_pred, y_pred = forward_kinematics(theta1, theta2, L1, L2)
    
    err = np.hypot(x_true - x_pred, y_true - y_pred)

    return err


In [ ]:
print(error(120, 60)) # error should be 0


In [ ]:
# constants
L1, L2 = 100, 100
N = 10_000
rng = np.random.default_rng(seed=42)

# joint angles
theta1 = rng.uniform(-np.pi, np.pi, N)
theta2 = rng.uniform(0, np.pi, N)


In [ ]:
x, y = forward_kinematics(theta1, theta2, L1, L2)
X = np.column_stack([x, y])
Y = np.column_stack([theta1, theta2])

print(f"X shape: {X.shape}, Y Shape: {Y.shape}")
print(X[0]) # sanity check
print(Y[0]) # sanity check


In [ ]:
rng = np.random.default_rng(seed=42)
indices = rng.permutation(N) # shuffle the 10,000 indexes randomly

train_end = int(0.70 * N) # 7000
val_end = int(0.85 * N) # 8500

train_idx = indices[:train_end] # first 7000 -> train
val_idx = indices[train_end:val_end] # next 1500 -> val
test_idx = indices[val_end:] # -> next 1500 -> test

X_train, Y_train = X[train_idx], Y[train_idx]
X_val, Y_val = X[val_idx], Y[val_idx]
X_test, Y_test = X[test_idx], Y[test_idx]

print(X_train.shape, Y_train.shape)
print(X_val.shape, Y_val.shape)
print(X_test.shape, Y_test.shape)


In [ ]:
# mean_error_calc is imported from planar_ik.metrics.


In [ ]:
# fit_linear and predict_linear are imported from planar_ik.models.


In [ ]:
W = fit_linear(X_train, Y_train)

Y_pred_train = predict_linear(W, X_train)
Y_pred_val = predict_linear(W, X_val)
Y_pred_test = predict_linear(W, X_test)

train_err = mean_error_calc(X_train, Y_pred_train, L1, L2)
val_err = mean_error_calc(X_val,   Y_pred_val,   L1, L2)
test_err = mean_error_calc(X_test,  Y_pred_test,  L1, L2)

print(f"Linear baseline mean EE error")
print(f"train: {train_err:.3f} mm")
print(f"val: {val_err:.3f} mm")
print(f"test: {test_err:.3f} mm")


In [ ]:
# ee_errors, summarize_errors, and print_summary are imported from planar_ik.


In [ ]:
print_summary(summarize_errors("Linear baseline train: ", X_train, Y_pred_train, L1, L2))
print("\n")
print_summary(summarize_errors("Linear baseline val: ", X_val, Y_pred_val, L1, L2))
print("\n")
print_summary(summarize_errors("Linear baseline test: ", X_test, Y_pred_test, L1, L2))


In [ ]:
# Compare true vs predicted end-effector positions for the linear baseline

Y_pred = Y_pred_val

x_true = X_val[:, 0]
y_true = X_val[:, 1]

theta1_pred = Y_pred[:, 0]
theta2_pred = Y_pred[:, 1]

x_pred, y_pred = forward_kinematics(theta1_pred, theta2_pred, L1, L2)

plt.figure(figsize=(7, 7))
plt.scatter(x_true, y_true, s=4, alpha=0.25, label="True targets")
plt.scatter(x_pred, y_pred, s=4, alpha=0.25, label="Linear predictions")
plt.axis("equal")
plt.xlabel("x position (mm)")
plt.ylabel("y position (mm)")
plt.title("Raw Linear Regression Fails on 2-DOF Inverse Kinematics")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# standardize_train_val_test and polynomial_features are imported from planar_ik.


In [ ]:
X_train_s, X_val_s, X_test_s, x_mean, x_std = standardize_train_val_test(
    X_train, X_val, X_test
)

degrees = list(range(1, 44))

weight_norms = []
condition_numbers = []
train_errors = []
val_errors = []

for degree in degrees:
    Phi_train = polynomial_features(X_train_s, degree)
    Phi_val = polynomial_features(X_val_s, degree)

    W_poly = fit_linear(Phi_train, Y_train)

    weight_norm = np.linalg.norm(W_poly)
    condition_number = np.linalg.cond(Phi_train.T @ Phi_train)

    weight_norms.append(weight_norm)
    condition_numbers.append(condition_number)

    Y_pred_train = predict_linear(W_poly, Phi_train)
    Y_pred_val = predict_linear(W_poly, Phi_val)

    train_err = mean_error_calc(X_train, Y_pred_train, L1, L2)
    val_err = mean_error_calc(X_val, Y_pred_val, L1, L2)

    train_errors.append(train_err)
    val_errors.append(val_err)

    print(f"degree {degree}: train = {train_err:.3f} mm, val = {val_err:.3f} mm")


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 12), sharex=True)

axes[0].plot(degrees, val_errors, marker="o")
axes[0].set_ylabel("Val EE error (mm)")
axes[0].set_title("Validation Error vs Polynomial Degree")
axes[0].grid(True)

axes[1].plot(degrees, weight_norms, marker="o")
axes[1].set_ylabel("||W_poly||")
axes[1].set_yscale("log")
axes[1].set_title("Weight Norm vs Polynomial Degree")
axes[1].grid(True)

axes[2].plot(degrees, condition_numbers, marker="o")
axes[2].set_xlabel("Polynomial degree")
axes[2].set_ylabel("cond(PhiÃ¡Âµâ‚¬ Phi)")
axes[2].set_yscale("log")
axes[2].set_title("Condition Number vs Polynomial Degree")
axes[2].grid(True)

plt.tight_layout()
plt.show()


## Ridge regression (L2 regularization)

Unregularized least squares solves $W = (\Phi^\top \Phi)^{-1} \Phi^\top Y$. As the polynomial degree grows, $\Phi^\top \Phi$ becomes ill-conditioned and $\|W\|$ blows up, exactly what we saw above past degree ~25.

Ridge adds $\lambda I$ to the Gram matrix:

$$W = (\Phi^\top \Phi + \lambda I)^{-1} \Phi^\top Y$$

We do **not** penalize the bias column (zero in the first diagonal entry of the identity).


In [ ]:
# fit_ridge and predict_ridge are imported from planar_ik.models.


### Sweep $\lambda$ at a fixed high degree

Pick a degree well past where the unregularized fit starts to degrade (degree 35 â€” already overfitting in the sweep above) and search a log-spaced grid of $\lambda$ values. Best $\lambda$ is chosen on the val set; test is reported once at the end.


In [ ]:
ridge_degree = 35

Phi_train_r = polynomial_features(X_train_s, ridge_degree)
Phi_val_r = polynomial_features(X_val_s, ridge_degree)
Phi_test_r = polynomial_features(X_test_s, ridge_degree)

lambdas = np.logspace(-6, 4, 30)

ridge_train_errs = []
ridge_val_errs = []
ridge_weight_norms = []

for lam in lambdas:
    W_r = fit_ridge(Phi_train_r, Y_train, lam)

    Y_pred_train_r = predict_ridge(W_r, Phi_train_r)
    Y_pred_val_r = predict_ridge(W_r, Phi_val_r)

    ridge_train_errs.append(mean_error_calc(X_train, Y_pred_train_r, L1, L2))
    ridge_val_errs.append(mean_error_calc(X_val, Y_pred_val_r, L1, L2))
    ridge_weight_norms.append(np.linalg.norm(W_r))

best_idx = int(np.argmin(ridge_val_errs))
best_lam = lambdas[best_idx]

print(f"degree = {ridge_degree}")
print(f"best lambda = {best_lam:.3e}")
print(f"best val error = {ridge_val_errs[best_idx]:.3f} mm")
print(f"unregularized val error at this degree = {val_errors[ridge_degree - 1]:.3f} mm")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True)

axes[0].plot(lambdas, ridge_train_errs, marker="o", label="train")
axes[0].plot(lambdas, ridge_val_errs, marker="o", label="val")
axes[0].axvline(best_lam, color="k", linestyle="--", alpha=0.5, label=f"best lambda = {best_lam:.1e}")
axes[0].set_xscale("log")
axes[0].set_ylabel("EE error (mm)")
axes[0].set_title(f"Ridge: EE error vs lambda (degree {ridge_degree})")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(lambdas, ridge_weight_norms, marker="o")
axes[1].axvline(best_lam, color="k", linestyle="--", alpha=0.5)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("lambda")
axes[1].set_ylabel("||W||")
axes[1].set_title("Weight norm vs lambda")
axes[1].grid(True)

plt.tight_layout()
plt.show()


### Final test evaluation at best (degree, lambda)


In [ ]:
W_best = fit_ridge(Phi_train_r, Y_train, best_lam)

Y_pred_train_best = predict_ridge(W_best, Phi_train_r)
Y_pred_val_best = predict_ridge(W_best, Phi_val_r)
Y_pred_test_best = predict_ridge(W_best, Phi_test_r)

print_summary(summarize_errors(f"Ridge (degree {ridge_degree}, lambda {best_lam:.2e}) train:", X_train, Y_pred_train_best, L1, L2))
print()
print_summary(summarize_errors(f"Ridge (degree {ridge_degree}, lambda {best_lam:.2e}) val:", X_val, Y_pred_val_best, L1, L2))
print()
print_summary(summarize_errors(f"Ridge (degree {ridge_degree}, lambda {best_lam:.2e}) test:", X_test, Y_pred_test_best, L1, L2))


## 2D grid search over (degree, $\lambda$)

The single-degree sweep showed ridge rescuing an overfit fit but not beating the unregularized degree-25 baseline. The real win comes from sweeping both axes jointly: a higher-degree basis *with* ridge regularization usually outperforms either alone.


In [ ]:
grid_degrees = list(range(15, 46))
grid_lambdas = np.logspace(-6, 4, 25)

grid_val = np.full((len(grid_degrees), len(grid_lambdas)), np.nan)

# precompute features per degree so we don't rebuild them inside the lambda loop
for i, degree in enumerate(grid_degrees):
    Phi_tr = polynomial_features(X_train_s, degree)
    Phi_va = polynomial_features(X_val_s, degree)

    for j, lam in enumerate(grid_lambdas):
        W_g = fit_ridge(Phi_tr, Y_train, lam)
        Y_pred_va = predict_ridge(W_g, Phi_va)
        grid_val[i, j] = mean_error_calc(X_val, Y_pred_va, L1, L2)

# best joint cell
flat_idx = int(np.nanargmin(grid_val))
best_i, best_j = np.unravel_index(flat_idx, grid_val.shape)
best_degree_grid = grid_degrees[best_i]
best_lam_grid = grid_lambdas[best_j]
best_val_grid = grid_val[best_i, best_j]

print(f"best (degree, lambda) = ({best_degree_grid}, {best_lam_grid:.3e})")
print(f"best val error = {best_val_grid:.3f} mm")
print(f"unregularized best (degree 25) val = {val_errors[24]:.3f} mm")
print(f"degree-35 ridge val = {ridge_val_errs[best_idx]:.3f} mm")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

im = ax.imshow(
    grid_val,
    aspect="auto",
    origin="lower",
    extent=[np.log10(grid_lambdas[0]), np.log10(grid_lambdas[-1]),
            grid_degrees[0], grid_degrees[-1]],
    cmap="viridis",
)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Val EE error (mm)")

ax.scatter(
    [np.log10(best_lam_grid)], [best_degree_grid],
    color="red", marker="x", s=120, linewidths=3,
    label=f"best: deg {best_degree_grid}, lam {best_lam_grid:.1e} -> {best_val_grid:.2f} mm",
)
ax.set_xlabel("log10(lambda)")
ax.set_ylabel("Polynomial degree")
ax.set_title("Val EE error over (degree, lambda) grid")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


### Final test evaluation at the joint optimum


In [ ]:
Phi_train_g = polynomial_features(X_train_s, best_degree_grid)
Phi_val_g = polynomial_features(X_val_s, best_degree_grid)
Phi_test_g = polynomial_features(X_test_s, best_degree_grid)

W_grid = fit_ridge(Phi_train_g, Y_train, best_lam_grid)

Y_pred_train_g = predict_ridge(W_grid, Phi_train_g)
Y_pred_val_g = predict_ridge(W_grid, Phi_val_g)
Y_pred_test_g = predict_ridge(W_grid, Phi_test_g)

tag = f"Ridge (deg {best_degree_grid}, lam {best_lam_grid:.2e})"
print_summary(summarize_errors(f"{tag} train:", X_train, Y_pred_train_g, L1, L2))
print()
print_summary(summarize_errors(f"{tag} val:",   X_val,   Y_pred_val_g,   L1, L2))
print()
print_summary(summarize_errors(f"{tag} test:",  X_test,  Y_pred_test_g,  L1, L2))


## Irreducible error floor and the case for MLPs


In [ ]:
floor_degree = 29
floor_lam = 3.162277660168379e-4

Phi_train_floor = polynomial_features(X_train_s, floor_degree)
Phi_test_floor = polynomial_features(X_test_s, floor_degree)
W_floor = fit_ridge(Phi_train_floor, Y_train, floor_lam)
Y_pred_floor = predict_ridge(W_floor, Phi_test_floor)

residual_ee = ee_errors(X_test, Y_pred_floor, L1, L2)
r = np.hypot(X_test[:, 0], X_test[:, 1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(residual_ee, bins=40, density=True, alpha=0.75, color="tab:blue")
axes[0].set_xlabel("EE residual error (mm)")
axes[0].set_ylabel("Density")
axes[0].set_title("Residual EE Error Distribution")
axes[0].grid(True, alpha=0.25)

sorted_err = np.sort(residual_ee)
ecdf = np.arange(1, len(sorted_err) + 1) / len(sorted_err)
axes[1].plot(sorted_err, ecdf, color="tab:orange", linewidth=2)
axes[1].set_xlabel("EE residual error (mm)")
axes[1].set_ylabel("ECDF")
axes[1].set_title("Residual EE Error ECDF")
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(r, residual_ee, s=8, alpha=0.25, label="test samples")

bins = np.linspace(r.min(), r.max(), 24)
bin_idx = np.digitize(r, bins)
bin_centers = []
bin_means = []
for i in range(1, len(bins)):
    mask = bin_idx == i
    if np.any(mask):
        bin_centers.append(r[mask].mean())
        bin_means.append(residual_ee[mask].mean())

ax.plot(bin_centers, bin_means, color="black", linewidth=2, label="radial mean")
ax.set_xlabel("Distance from origin r (mm)")
ax.set_ylabel("EE residual error (mm)")
ax.set_title("Structured Residual Error Across the Workspace")
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

print(f"mean residual: {residual_ee.mean():.2f} mm")
print(f"median residual: {np.median(residual_ee):.2f} mm")


Polynomial ridge bottoms out around ~19 mm mean / ~7 mm median EE error. The remaining error is structured (concentrated near the workspace boundary and near singular configurations), which is exactly the regime where a learned nonlinear feature map should help. The next iteration of this project replaces polynomial lifting with an MLP.
